In [1]:
import json
import os
from pathlib import Path

import tiktoken
from openai import OpenAI
from dotenv import load_dotenv

env_path = Path("/home/luis/Documents/FGV/Laboratory/document-graph/server/.env")
load_dotenv(dotenv_path=env_path)

MODELLM = "gpt-4.1-mini"
APIKEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=APIKEY)

INPUT_PATH = Path("../../infra/json/kg_extraction/parse.json")
OUTPUT_PATH = Path("../../infra/json/kg_extraction/kg.json")

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

# print("Input keys:", data.keys())
# print("Total nodes:", len(data.get("nodes", [])))


In [2]:
ALLOWED_NODE_TYPES = [
    # "Clause",
    "DefinedTerm",
    "Party",
    "Obligation",
    "Right",
    "Permission",
    "Prohibition",
    "Condition",
    "Reference",
    "Value",
]

ALLOWED_EDGE_TYPES = [
    "IS_PART_OF",
    # "CONTAINS",
    # "REFERENCES",
    "DEFINES",
    "USES",
    "ASSIGNS_OBLIGATION_TO",
    "GRANTS_RIGHT_TO",
    "DEPENDS_ON",
    "MODIFIES",
    "AMENDS",
    "SUPERSEDES",
    "CONTRADICTS",
    "CONTRADICCION",
    "CONTRADICTION",
]

SEED_EDGE_TYPES = {"REFERENCES"}



In [3]:
def build_seed_graph(paragraph_rows):
    nodes = []
    edges = []
    seen_nodes = set()
    seen_edges = set()

    for row in paragraph_rows:
        src_id = str(row["paragraph_id"])
        src_text = str(row.get("text", "")).strip()

        if src_id not in seen_nodes:
            nodes.append({
                "id": src_id,
                "type": "Clause",
                "label": src_id,
                "text": src_text,
                "properties": {},
                "evidence_text": src_text
            })
            seen_nodes.add(src_id)

        for rel in row.get("related_paragraphs", []):
            tgt_id = str(rel["paragraph_id"])
            tgt_text = str(rel.get("text", "")).strip()

            if tgt_id not in seen_nodes:
                nodes.append({
                    "id": tgt_id,
                    "type": "Clause",
                    "label": tgt_id,
                    "text": tgt_text,
                    "properties": {},
                    "evidence_text": tgt_text
                })
                seen_nodes.add(tgt_id)

            ekey = (src_id, "REFERENCES", tgt_id)
            if ekey not in seen_edges:
                edges.append({
                    "source": src_id,
                    "target": tgt_id,
                    "type": "REFERENCES",
                    "evidence_text": src_text,
                    "source_field": None,
                    "target_field": None
                })
                seen_edges.add(ekey)

    return {"nodes": nodes, "edges": edges}


In [4]:
seed_kg = build_seed_graph(data)

In [5]:
def make_extraction_prompt(clause_row):
    clause_id = str(clause_row["paragraph_id"])
    clause_text = str(clause_row.get("text", "")).strip()
    related = clause_row.get("related_paragraphs", [])

    SYSTEM_PROMPT = """You will receive the full details of clauses: {id, text, title, level}."""

    PROMPT_INSTRUCTION = """
Your task is to act as a legal graph extractor. From clauses, create a contained set of nodes and edges that are explicitly supported by the text. Follow the reasoning process, rules, and clarifications below.

Output, strict JSON object with this structure:

{
 "contract_id": "...",
 "nodes": [ ... ],
 "edges": [ ... ]
}

──────────────────────────
REASONING PROCESS
──────────────────────────
1.  **Isolate Core Text:** First, mentally separate the core contractual prose from any 'noise' like Tables of Contents, redaction headers, or formatting artifacts. Your analysis should ONLY focus on the contractual prose.
2.  **Create Primary Node:** Create the `CLAUSE` node for the clause you were given.
3.  **Infer and Create Parent Node:** Analyze the clause `id` to infer the parent clause ID. Create the parent `CLAUSE` node and the IS_PART_OF edge to it.
4.  **Scan and Create Nodes:** Read the core text to identify all other entities (Referenced Clauses, Defined Terms, Parties, Values) and create their corresponding nodes according to the rules below.
5.  **Create Edges:** Link the primary clause node to all other created nodes using the appropriate edge types.

──────────────────────────
NODE RULES
──────────────────────────
- CLAUSE:
  { "id": "<clause-id>", "node_type": "CLAUSE", "title": "<title>", "level": <int> }
  Rule: Create a node for the input clause, its inferred parent, and any clauses it explicitly references.

- DEFINED_TERM:
  { "id": "term:<Canonical Term>", "node_type": "DEFINED_TERM", "name": "<Canonical Term>" }
  Rule: Create for terms with special meaning (quoted, ALL CAPS, or explicitly defined). Canonicalize the name (e.g., "this Agreement" becomes "Agreement"; use Title Case).

- PARTY:
  { "id": "party:<Party Name>", "node_type": "PARTY", "name": "<Party Name>" }
  Rule: Only for legal entities or defined roles (e.g., "Licensor", "the Supplier"). Canonicalize by removing articles ("the", "a").

- VALUE:
  { "id": "value:<literal text>", "node_type": "VALUE", "unit": "Currency|Percentage|Days|Months|Years", "text": "<literal text>" }
  Rule: Extract specific amounts, durations, etc.

──────────────────────────
EDGE RULES
──────────────────────────
Format: { "src":"<id>", "tgt":"<id>", "type":"<EDGE_TYPE>" }

- IS_PART_OF: CLAUSE → parent CLAUSE
  Rule: Infer parent from child's ID. Numeric: "3.4" → "3"; "14.2" → "ARTICLE 14". Non-numeric: For an ID like "(h)", look for context in the text like "Section 3.2(h)" to infer the parent is "3.2".

- DEFINES: CLAUSE → DEFINED_TERM
  Rule: Use when the clause introduces a definition (“X shall mean...”, “(the ‘X’)”).

- USES: CLAUSE → DEFINED_TERM
  Rule: Use when a defined term is mentioned but not defined in this clause. Be thorough and include all capitalized, multi-word legal concepts (e.g., "Material Change", "Product Prices").

- REFERENCES: CLAUSE → CLAUSE
  Rule: Use for explicit cross-references like “Section 3.2” or “Article 10”.

- MENTIONS_PARTY: CLAUSE → PARTY
  Rule: Create an edge for every mentioned party.

- CONTAINS: CLAUSE → VALUE
  Rule: Create an edge for every extracted value.

- CONTRADICTS: CLAUSE/ENTITY → CLAUSE/ENTITY
  Rule: Create this edge only when two statements are materially incompatible under comparable scope/condition.
  Examples: one says an action is required and another says it is not required; different thresholds/time windows trigger opposite outcomes (e.g., 6 months vs 3 months for the same remedy logic).
  Prefer entity-level contradiction when possible (Condition↔Condition, Obligation↔Obligation, Value↔Value); otherwise use Clause↔Clause.
  Use RELATED_PARAGRAPHS_JSON to detect cross-clause contradictions.
  Do not create CONTRADICTS for mere differences in wording, specificity, or non-conflicting exceptions.


──────────────────────────
CRITICAL CLARIFICATIONS
──────────────────────────
1.  **Reference Typing is Key:** Any cross-reference to another part of the document (e.g., "Article 5", "Exhibit B") MUST be created as a `CLAUSE` node. It is NEVER a `DEFINED_TERM`.
2.  **No Duplicates:** Output each unique node and edge only once. If no nodes or edges can be created, output empty arrays.
3.  **Sort for Consistency:** Sort nodes by ID and edges by `src`, `type`, then `tgt`."""


    clause_payload = {
        "id": clause_id,
        "text": clause_text,
        "title": str(clause_row.get("title", "")),
        "level": clause_row.get("level", None),
    }

    prompt = (
        f"{SYSTEM_PROMPT}\n\n"
        f"{PROMPT_INSTRUCTION}\n\n"
        "CLAUSE_INPUT_JSON:\n"
        f"{json.dumps(clause_payload, ensure_ascii=False, indent=2)}\n\n"
        "RELATED_PARAGRAPHS_JSON:\n"
        f"{json.dumps(related, ensure_ascii=False, indent=2)}"
    )

    return prompt




In [6]:
enc = tiktoken.encoding_for_model(MODELLM)

def safe_json_loads(text):
    text = text.strip()

    if text.startswith("```json"):
        text = text.removeprefix("```json").removesuffix("```").strip()
    elif text.startswith("```"):
        text = text.removeprefix("```").removesuffix("```").strip()

    start = text.find("{")
    end = text.rfind("}")

    if start == -1 or end == -1:
        raise ValueError("No JSON object found in model output.")

    return json.loads(text[start:end + 1])

def extract_kg_from_clause(clause_row):
    prompt = make_extraction_prompt(clause_row)
    input_tokens = len(enc.encode(prompt))

    response = client.responses.create(
        model=MODELLM,
        input=prompt,
        temperature=0,
    )

    output_text = response.output_text
    output_tokens = len(enc.encode(output_text))
    kg_json = safe_json_loads(output_text)
    return kg_json, input_tokens, output_tokens

# # prompt = extract_kg_from_clause(client, nodes)
# kg_json, in_t, out_t = extract_kg_from_clause(client, data)

In [7]:
def normalize_and_validate_part(clause_id, kg_part):
    entities = kg_part.get("entities", kg_part.get("nodes", []))
    relations = kg_part.get("relations", kg_part.get("edges", []))

    node_type_alias = {
        "DEFINED_TERM": "DefinedTerm",
        "PARTY": "Party",
        "OBLIGATION": "Obligation",
        "RIGHT": "Right",
        "PERMISSION": "Permission",
        "PROHIBITION": "Prohibition",
        "CONDITION": "Condition",
        "REFERENCE": "Reference",
        "VALUE": "Value",
    }

    clean_entities = []
    entity_ids = set()

    for e in entities:
        if not isinstance(e, dict):
            continue

        raw_etype = str(e.get("type", e.get("node_type", ""))).strip()
        etype = node_type_alias.get(raw_etype, raw_etype)
        eid = str(e.get("id", "")).strip()
        if not etype or not eid:
            continue
        if etype not in ALLOWED_NODE_TYPES:
            continue

        if not eid.startswith(f"{clause_id}::"):
            eid = f"{clause_id}::{eid}"
        if eid in entity_ids:
            continue

        clean_entities.append({
            "id": eid,
            "type": etype,
            "label": str(e.get("label", e.get("name", e.get("title", eid)))),
            "clause_id": clause_id,
            "properties": e.get("properties", {}) if isinstance(e.get("properties"), dict) else {},
            "evidence_text": str(e.get("evidence_text", e.get("text", ""))).strip(),
        })
        entity_ids.add(eid)

    local_to_full = {e["id"].split("::", 1)[-1]: e["id"] for e in clean_entities}

    clean_relations = []
    seen_rel = set()

    for r in relations:
        if not isinstance(r, dict):
            continue

        rtype = str(r.get("type", "")).strip().upper()
        if rtype in {"CONTRADICCION", "CONTRADICTION"}:
            rtype = "CONTRADICTS"
        if rtype not in ALLOWED_EDGE_TYPES:
            continue

        src = str(r.get("source", r.get("src", ""))).strip()
        tgt = str(r.get("target", r.get("tgt", ""))).strip()

        if src in local_to_full:
            src = local_to_full[src]
        if tgt in local_to_full:
            tgt = local_to_full[tgt]

        if rtype == "CONTRADICTS":
            # Allow contradictions between any ids (clause<->clause, entity<->entity, clause<->entity)
            if not src or not tgt:
                continue
        else:
            valid_target_clause = (rtype == "IS_PART_OF" and tgt == clause_id)
            if src not in entity_ids:
                continue
            if not (tgt in entity_ids or valid_target_clause):
                continue

        rkey = (src, rtype, tgt)
        if rkey in seen_rel:
            continue

        clean_relations.append({
            "source": src,
            "target": tgt,
            "type": rtype,
            "evidence_text": str(r.get("evidence_text", "")).strip(),
            "source_field": r.get("source_field", None),
            "target_field": r.get("target_field", None),
        })
        seen_rel.add(rkey)

    return {"entities": clean_entities, "relations": clean_relations}



In [8]:
def enforce_is_part_of(clause_id, kg_part):
    rel_set = {(r["source"], r["type"], r["target"]) for r in kg_part["relations"]}
    for ent in kg_part["entities"]:
        key = (ent["id"], "IS_PART_OF", clause_id)
        if key not in rel_set:
            kg_part["relations"].append({
                "source": ent["id"],
                "target": clause_id,
                "type": "IS_PART_OF",
                "evidence_text": ent.get("evidence_text", ""),
                "source_field": None,
                "target_field": None,
            })
            rel_set.add(key)
    return kg_part


def merge_graph(seed_graph, all_parts):
    nodes = list(seed_graph["nodes"])
    edges = list(seed_graph["edges"])

    seen_node_ids = {n["id"] for n in nodes}
    seen_edge_keys = {(e["source"], e["type"], e["target"]) for e in edges}

    for part in all_parts:
        for e in part["entities"]:
            if e["id"] not in seen_node_ids:
                nodes.append(e)
                seen_node_ids.add(e["id"])

        for r in part["relations"]:
            k = (r["source"], r["type"], r["target"])
            if k not in seen_edge_keys:
                edges.append(r)
                seen_edge_keys.add(k)

    return {"nodes": nodes, "edges": edges}


def expand_rows_with_related(paragraph_rows):
    expanded = []
    seen = set()

    for row in paragraph_rows:
        row_id = str(row["paragraph_id"])
        if row_id not in seen:
            expanded.append(row)
            seen.add(row_id)

        for rel in row.get("related_paragraphs", []):
            rel_id = str(rel["paragraph_id"])
            if rel_id in seen:
                continue
            expanded.append({
                "paragraph_id": rel_id,
                "text": str(rel.get("text", "")),
                "related_paragraphs": [{
                    "paragraph_id": row_id,
                    "text": str(row.get("text", "")),
                }],
            })
            seen.add(rel_id)

    return expanded



In [9]:
with open(INPUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

expanded_data = expand_rows_with_related(data)
seed_graph = build_seed_graph(data)

all_parts = []
total_in = 0
total_out = 0

for i, clause_row in enumerate(expanded_data, start=1):
    clause_id = str(clause_row["paragraph_id"])
    raw_part, in_t, out_t = extract_kg_from_clause(clause_row)
    total_in += in_t
    total_out += out_t

    part = normalize_and_validate_part(clause_id, raw_part)
    part = enforce_is_part_of(clause_id, part)
    all_parts.append(part)

    if i % 10 == 0:
        print(f"[{i}/{len(expanded_data)}] ok")

final_graph = merge_graph(seed_graph, all_parts)

output_payload = {
    "stats": {
        "clauses_input": len(data),
        "clauses_processed": len(expanded_data),
        "seed_nodes": len(seed_graph["nodes"]),
        "seed_edges": len(seed_graph["edges"]),
        "final_nodes": len(final_graph["nodes"]),
        "final_edges": len(final_graph["edges"]),
        "input_tokens": total_in,
        "output_tokens": total_out,
    },
    "graph": final_graph
}

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(output_payload, f, ensure_ascii=False, indent=2)

print(f"Saved: {OUTPUT_PATH}")
print(json.dumps(output_payload["stats"], indent=2))




Saved: ../../infra/json/kg_extraction/kg.json
{
  "clauses_input": 1,
  "clauses_processed": 2,
  "seed_nodes": 2,
  "seed_edges": 1,
  "final_nodes": 19,
  "final_edges": 19,
  "input_tokens": 3014,
  "output_tokens": 2180
}
